# 04 Neurosymbolic Inference Heatmap

Interactive notebook for neurosymbolic inference with heatmap explanations from uploaded PCB images.

- Faster R-CNN provides backbone, RPN, RoI Align, bbox regressor, and detection post-processing.
- SODT replaces the classification branch and receives the flattened RoI Align pooled grid `[C, 7, 7]`.
- Explanations display **Local Evidence per decision node**, not a combined path heatmap.


In [ ]:
from pathlib import Path
import io

import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import clear_output, display
from PIL import Image

from notebooks.util import resolve_root
from neuro.config import NeuroConfig, NeuroTrainConfig
from neuro.preprocess_dataset import test_preprocess
from neurosym.inference import (
    explain_hybrid_detection,
    load_neurosymbolic_detector,
    run_neurosymbolic_inference,
    select_detection_indices,
)
from neurosym.visualization import (
    draw_neurosymbolic_explanation,
    draw_numbered_detections,
)
from util.artifacts import latest_run_checkpoint
from util.config import load_yaml
from util.device import select_device

PROJECT_ROOT = resolve_root()

In [ ]:
neuro_config = load_yaml(Path("neuro.yaml"), NeuroConfig)
train_config = load_yaml(Path("neuro_train.yaml"), NeuroTrainConfig)

device = select_device(train_config["device"])
detector_checkpoint_path = latest_run_checkpoint(PROJECT_ROOT / "checkpoints" / "neuro")
symbolic_checkpoint_path = latest_run_checkpoint(PROJECT_ROOT / "checkpoints" / "symbolic")

hybrid_model, detector_checkpoint = load_neurosymbolic_detector(
    detector_checkpoint_path=detector_checkpoint_path,
    neuro_config=neuro_config,
    train_config=train_config,
    symbolic_checkpoint_path=symbolic_checkpoint_path,
    device=str(device),
)

image_preprocess = test_preprocess()
class_names = tuple(train_config["dataset"]["class_names"])

detector_checkpoint_path, symbolic_checkpoint_path

In [ ]:
def class_name(label: int) -> str:
    return class_names[int(label) - 1]


In [ ]:
upload_widget = widgets.FileUpload(
    accept="image/*",
    multiple=False,
    description="Upload Image",
)
MAX_DISPLAY_DETECTIONS = 9
DISPLAY_SCORE_THRESHOLD = 0.3
detection_output = widgets.Output()
explanation_output = widgets.Output()
state = {
    "image_name": None,
    "image_tensor": None,
    "detection": None,
    "selected_indices": [],
}


def uploaded_file_record():
    value = upload_widget.value
    if isinstance(value, tuple):
        return value[0] if value else None
    if isinstance(value, dict):
        if "content" in value:
            return value
        return next(iter(value.values())) if value else None
    return None


def uploaded_content_bytes(file_record) -> bytes:
    content = file_record["content"]
    return content.tobytes() if isinstance(content, memoryview) else bytes(content)


def render_detection(detection_index: int) -> None:
    image_tensor = state["image_tensor"]
    detection = state["detection"]
    selected_indices = state["selected_indices"]
    explanation = explain_hybrid_detection(
        hybrid_model,
        detection,
        detection_index=detection_index,
        image_shape=tuple(image_tensor.shape[-2:]),
    )
    with explanation_output:
        clear_output(wait=True)
        selected_number = selected_indices.index(detection_index) + 1
        draw_neurosymbolic_explanation(
            image_tensor,
            detection,
            detection_index,
            explanation,
            class_names,
            hybrid_model.symbolic_tree,
            selected_number=selected_number,
        )


def make_detection_button(display_number: int, detection_index: int) -> widgets.Button:
    detection = state["detection"]
    label = int(detection["labels"][detection_index])
    score = float(detection["scores"][detection_index])
    button = widgets.Button(
        description=f"#{display_number} {class_name(label)} {score:.2f}",
        layout=widgets.Layout(width="180px"),
    )
    button.on_click(lambda _: render_detection(detection_index))
    return button


def run_uploaded_inference(_button) -> None:
    file_record = uploaded_file_record()
    with detection_output:
        clear_output(wait=True)
        explanation_output.clear_output(wait=True)

        if file_record is None:
            print("Upload one PCB image first.")
            return

        image_name = file_record.get("name", "uploaded_image")
        pil_image = Image.open(io.BytesIO(uploaded_content_bytes(file_record))).convert(
            "RGB"
        )
        image_tensor = image_preprocess(pil_image)
        detection = run_neurosymbolic_inference(hybrid_model, [image_tensor])[0]
        selected_indices = select_detection_indices(
            detection,
            score_threshold=DISPLAY_SCORE_THRESHOLD,
            max_detections=MAX_DISPLAY_DETECTIONS,
        )

        state["image_name"] = image_name
        state["image_tensor"] = image_tensor
        state["detection"] = detection
        state["selected_indices"] = selected_indices

        fig, axis = plt.subplots(figsize=(8, 8))
        draw_numbered_detections(axis, image_tensor, detection, selected_indices, class_names)
        axis.set_title(f"Neuro-Symbolic detections: {image_name}")
        plt.show()

        if not selected_indices:
            print("No detections were returned by the model.")
            return

        buttons = [
            make_detection_button(display_number, detection_index)
            for display_number, detection_index in enumerate(selected_indices, start=1)
        ]
        display(
            widgets.GridBox(
                buttons,
                layout=widgets.Layout(
                    grid_template_columns="repeat(3, 190px)",
                    grid_gap="8px",
                ),
            )
        )

        render_detection(selected_indices[0])


def run_uploaded_inference_on_upload(change) -> None:
    if change["new"]:
        run_uploaded_inference(None)


upload_widget.observe(run_uploaded_inference_on_upload, names="value")
display(
    widgets.VBox(
        [
            upload_widget,
            detection_output,
            explanation_output,
        ]
    )
)